# 07 — From one field to a distribution of catalogs

A coarse field does not specify a unique unresolved catalog. The
target is a conditional marked point process:

$$
N\sim p(N\mid F),\qquad
\mathcal C=\{(x_j,y_j,A_j)\}_{j=1}^{N}
\sim p(\mathcal C\mid N,F).
$$

We will build a small two-dimensional example. It is a teaching
model, not GOTHAM and not a halo finder. Its sections follow the
lecture's six choices: **represent, encode, mask, predict, sample,
validate**.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(os.path.abspath('.')).parent

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".mplconfig"))

SEED = 2603
COLORS = {
    "blue": "#2D6A9F",
    "orange": "#E6862E",
    "green": "#3A8D72",
    "purple": "#7656A5",
    "red": "#C94C4C",
    "gray": "#626C78",
}

def savefig(fig, name, evidence="analytic-fixture"):
    fig.text(0.995, 0.005, evidence, ha="right", va="bottom",
             fontsize=7, color=COLORS["gray"])
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=160, bbox_inches="tight", facecolor="white")
    print("saved:", path.relative_to(ROOT))


from dataclasses import dataclass
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

## 1. Represent — turn a catalog into a sentence

Each point becomes three tokens: `x`, `y`, and amplitude `A`.
Higher-amplitude tokens are written first; ties use x then y. Thus
the same quantized set has one reproducible order. `END` models
catalog length; `PAD` only makes arrays rectangular.

In [ ]:
@dataclass(frozen=True)
class ToyTokenSpec:
    field_size: int = 16
    n_bins: int = 8
    max_points: int = 3
    amplitude_min: float = 0.35
    amplitude_max: float = 1.00

    @property
    def START(self): return 3 * self.n_bins

    @property
    def END(self): return self.START + 1

    @property
    def PAD(self): return self.START + 2

    @property
    def vocab_size(self): return 3 * self.n_bins + 3

    @property
    def max_length(self): return 1 + 3 * self.max_points + 1


def tokenize_catalog(catalog, spec):
    '''Serialize points as START, (x,y,A)*, END, PAD.'''
    catalog = np.asarray(catalog, dtype=float).reshape(-1, 3)

    # TODO: map x, y, and amplitude into their three disjoint token ranges;
    # sort the token triples by amplitude, then x and y; assemble
    # START + body + END + PAD.
    raise NotImplementedError


def detokenize_catalog(sequence, spec):
    '''Decode each token to its bin center.'''
    sequence = np.asarray(sequence)
    end = int(np.flatnonzero(sequence == spec.END)[0])
    points = []
    for start in range(1, end, 3):
        x_token, y_token, a_token = sequence[start:start + 3]
        x = (x_token + 0.5) * spec.field_size / spec.n_bins
        y = (y_token - spec.n_bins + 0.5) * spec.field_size / spec.n_bins
        amplitude = spec.amplitude_min + (
            a_token - 2 * spec.n_bins + 0.5
        ) * (spec.amplitude_max - spec.amplitude_min) / spec.n_bins
        points.append([x, y, amplitude])
    return np.asarray(points, dtype=float).reshape(-1, 3)

In [ ]:
SPEC = ToyTokenSpec()
example_catalog = np.array([[2.2, 13.1, 0.55], [11.7, 4.4, 0.91]])
example_tokens = tokenize_catalog(example_catalog, SPEC)
example_decoded = detokenize_catalog(example_tokens, SPEC)
tie_catalog = np.array([[2.2, 13.1, 0.551], [11.7, 4.4, 0.549]])
tie_tokens = tokenize_catalog(tie_catalog, SPEC)
np.testing.assert_array_equal(
    tie_tokens,
    tokenize_catalog(tie_catalog[::-1], SPEC),
)
np.testing.assert_array_equal(
    tie_tokens,
    tokenize_catalog(detokenize_catalog(tie_tokens, SPEC), SPEC),
)

print("tokens:", example_tokens.tolist())
print("decoded bin centers:\n", np.round(example_decoded, 3))

## 2. Encode — make a known conditional point process

Smooth peaks form the observed field. The same field fixes the
Poisson rate, spatial probability, and amplitude distribution, but
every catalog draw remains stochastic. Counts are capped at three
only to keep the notebook small.

In [ ]:
def make_field(rng, size=SPEC.field_size):
    y, x = np.mgrid[0:size, 0:size] + 0.5
    field = np.zeros((size, size), dtype=np.float32)
    for _ in range(rng.integers(1, 4)):
        x0, y0 = rng.uniform(2, size - 2, size=2)
        amplitude = rng.uniform(0.7, 1.3)
        width = rng.uniform(1.0, 2.0)
        field += amplitude * np.exp(-((x - x0) ** 2 + (y - y0) ** 2) / (2 * width ** 2))
    return field + rng.normal(0, 0.025, field.shape).astype(np.float32)


def draw_catalog(field, rng, spec=SPEC):
    positive = np.clip(field, 0, None)
    rate = np.clip(0.35 + 0.035 * positive.sum(), 0.4, 2.5)
    count = min(int(rng.poisson(rate)), spec.max_points)
    if count == 0:
        return np.empty((0, 3))

    probability = 0.01 + positive ** 2
    probability /= probability.sum()
    pixels = rng.choice(spec.field_size ** 2, count, p=probability.ravel())
    iy, ix = np.unravel_index(pixels, field.shape)
    x = ix + rng.random(count)
    y = iy + rng.random(count)
    local = positive[iy, ix] / max(float(positive.max()), 1e-6)
    amplitude = np.clip(
        spec.amplitude_min + 0.60 * local + rng.normal(0, 0.05, count),
        spec.amplitude_min,
        spec.amplitude_max,
    )
    return np.column_stack([x, y, amplitude])


def make_dataset(size, seed):
    rng = np.random.default_rng(seed)
    fields, catalogs, tokens = [], [], []
    for _ in range(size):
        field = make_field(rng)
        catalog = draw_catalog(field, rng)
        fields.append(field)
        catalogs.append(catalog)
        tokens.append(tokenize_catalog(catalog, SPEC))
    return np.stack(fields), catalogs, np.stack(tokens)


# More independent fields help more than extra passes over a small stochastic set.
train_fields_np, train_catalogs, train_tokens_np = make_dataset(2048, SEED + 1)
val_fields_np, val_catalogs, val_tokens_np = make_dataset(512, SEED + 2)
field_mean, field_std = train_fields_np.mean(), train_fields_np.std()

def field_tensor(array):
    return torch.tensor(((array - field_mean) / field_std)[:, None], dtype=torch.float32)

train_fields = field_tensor(train_fields_np)
val_fields = field_tensor(val_fields_np)
train_tokens = torch.tensor(train_tokens_np, dtype=torch.long)
val_tokens = torch.tensor(val_tokens_np, dtype=torch.long)

In [ ]:
fixed_field = val_fields_np[7]
draw_rng = np.random.default_rng(SEED + 3)
repeated_catalogs = [draw_catalog(fixed_field, draw_rng) for _ in range(4)]

fig, axes = plt.subplots(1, 5, figsize=(12.8, 2.8), constrained_layout=True)
axes[0].imshow(
    fixed_field, origin="lower",
    extent=(0, SPEC.field_size, 0, SPEC.field_size), cmap="magma",
)
axes[0].set_title("fixed field")
for ax, catalog in zip(axes[1:], repeated_catalogs):
    ax.imshow(
        fixed_field, origin="lower",
        extent=(0, SPEC.field_size, 0, SPEC.field_size),
        cmap="Greys", alpha=0.3,
    )
    if len(catalog):
        ax.scatter(catalog[:, 0], catalog[:, 1], s=30 + 80 * catalog[:, 2],
                   c=catalog[:, 2], cmap="viridis", vmin=0.35, vmax=1.0)
    ax.set_title(f"draw: N={len(catalog)}")
for ax in axes:
    ax.set(
        xlim=(0, SPEC.field_size), ylim=(0, SPEC.field_size),
        xlabel="x", ylabel="y",
    )
    ax.set_aspect("equal")
    ax.grid(False)
savefig(fig, "07_toy_process.png")
plt.show()

## 3. Mask — connect a causal writer to field memory

A convolution projects the $16\times16$ field into sixteen
$4\times4$ patch-memory vectors. The decoder embeds its legal
prefix, uses masked self-attention, consults every field vector
through cross-attention, and applies a position-wise MLP. Each
sublayer writes an update to the residual state. This reader is a
patch projection, not a transformer encoder. Learned absolute
positions provide no built-in periodic, translation, or rotation
symmetry.

In [ ]:
def causal_mask(length, device):
    '''True entries are blocked by torch.nn.MultiheadAttention.'''
    # TODO: return a Boolean mask with only the strict upper triangle blocked.
    raise NotImplementedError



class TinyCatalogTransformer(nn.Module):
    '''A patch reader and one pre-norm transformer decoder block.'''

    def __init__(self, spec, d_model=32, n_heads=4, patch_size=4):
        super().__init__()
        self.spec = spec
        self.patch = nn.Conv2d(1, d_model, patch_size, stride=patch_size)
        n_patches = (spec.field_size // patch_size) ** 2
        self.field_position = nn.Parameter(torch.zeros(1, n_patches, d_model))
        self.token_embedding = nn.Embedding(spec.vocab_size, d_model)
        self.token_position = nn.Parameter(torch.zeros(1, spec.max_length - 1, d_model))
        self.self_attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.cross_attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.self_norm = nn.LayerNorm(d_model)
        self.cross_norm = nn.LayerNorm(d_model)
        self.mlp_norm = nn.LayerNorm(d_model)
        self.output_norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 2 * d_model),
            nn.GELU(),
            nn.Linear(2 * d_model, d_model),
        )
        self.output = nn.Linear(d_model, spec.vocab_size)

    def encode_field(self, field):
        patches = self.patch(field).flatten(2).transpose(1, 2)
        # TODO: add the learned field-position vectors to the patch memory.
        raise NotImplementedError

    def forward(self, field, tokens, return_attention=False):
        memory = self.encode_field(field)
        # TODO: (1) embed the prefix, add token positions, and mark PAD keys;
        # (2) pre-norm, causally self-attend, and add the residual update;
        # (3) pre-norm, cross-attend with state as Q and memory as K,V, then add;
        # (4) add the position-wise MLP update and project normalized logits.
        # Return (logits, cross-attention weights) only when requested.
        raise NotImplementedError

## 4. Predict — train on the next token

Teacher forcing shifts one complete sentence into `input` and
`target` rows. A causal mask and a one-token shift solve different
leakage paths. We compare with a position-only token baseline and
with the field/catalog pairing deliberately shuffled. Because one
stochastic target is stored per field, we restore the best
held-out checkpoint rather than trusting falling training loss.

In [ ]:
def sequence_loss(model, fields, sequences):
    '''Teacher-forced next-token cross-entropy.'''
    # TODO: shift the full sentence into input/target rows, run the model,
    # and compute cross-entropy while ignoring PAD targets.
    raise NotImplementedError

In [ ]:
def position_only_nll(reference, evaluation, alpha=0.5):
    '''Empirical next-token baseline with no prefix content and no field.'''
    length = reference.shape[1] - 1
    counts = np.full((length, SPEC.vocab_size), alpha, dtype=float)
    for position in range(length):
        target = reference[:, position + 1]
        valid = target != SPEC.PAD
        counts[position] += np.bincount(
            target[valid], minlength=SPEC.vocab_size
        )
    probabilities = counts / counts.sum(axis=1, keepdims=True)
    targets = evaluation[:, 1:]
    chosen = probabilities[np.arange(length)[None, :], targets]
    return float(-np.log(chosen[targets != SPEC.PAD]).mean())


torch.manual_seed(SEED + 4)
model = TinyCatalogTransformer(SPEC).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
train_history, validation_history, shuffled_history = [], [], []
best_state, best_validation_loss, best_epoch = None, np.inf, None

for epoch in range(30):
    model.train()
    order = torch.randperm(len(train_fields))
    epoch_losses = []
    for start in range(0, len(order), 64):
        index = order[start:start + 64]
        loss = sequence_loss(model, train_fields[index], train_tokens[index])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.detach()))
    train_history.append(float(np.mean(epoch_losses)))
    model.eval()
    with torch.no_grad():
        validation_loss = float(sequence_loss(model, val_fields, val_tokens))
        shuffled_loss = float(
            sequence_loss(model, val_fields.roll(1, 0), val_tokens)
        )
    validation_history.append(validation_loss)
    shuffled_history.append(shuffled_loss)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch + 1
        best_state = {
            name: value.detach().clone()
            for name, value in model.state_dict().items()
        }

model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    best_shuffled_loss = float(
        sequence_loss(model, val_fields.roll(1, 0), val_tokens)
    )
    original_prefix = val_tokens[:1, :-1]
    changed_prefix = original_prefix.clone()
    changed_prefix[:, 3:] = SPEC.START
    original_logits = model(val_fields[:1], original_prefix)
    changed_logits = model(val_fields[:1], changed_prefix)
torch.testing.assert_close(
    original_logits[:, :3], changed_logits[:, :3], atol=1e-6, rtol=1e-6
)
position_baseline = position_only_nll(train_tokens_np, val_tokens_np)

fig, ax = plt.subplots(figsize=(6.3, 3.5), constrained_layout=True)
epochs = np.arange(1, len(train_history) + 1)
ax.plot(epochs, train_history, label="train")
ax.plot(epochs, validation_history, label="validation")
ax.plot(epochs, shuffled_history, ls="--", label="validation; shuffled field")
ax.axhline(
    position_baseline, color=COLORS["gray"], ls=":", label="position-only baseline"
)
ax.axvline(
    best_epoch, color=COLORS["gray"], lw=1,
    label=f"restored epoch {best_epoch}",
)
ax.set(xlabel="epoch", ylabel="next-token cross-entropy [nats]",
       title="Teacher-forced training")
ax.legend()
savefig(fig, "07_training_diagnostics.png", "trained-teaching-model")
plt.show()
print(
    f"best epoch: {best_epoch} | validation: {best_validation_loss:.3f} | "
    f"shuffled field: {best_shuffled_loss:.3f} | "
    f"position-only: {position_baseline:.3f}"
)

## 5. Sample — grow a complete catalog

Generation starts from `START` and feeds each sampled token back to
the decoder. The grammar enforces property roles, legal stopping,
and the declared amplitude-then-coordinate order. It keeps the
sentence valid; it does not make the probabilities accurate.
Because the grammar is imposed at sampling, valid syntax is not
evidence that the network learned the grammar unaided.

In [ ]:
def allowed_next(prefix, spec):
    '''Legal next-token support, including the canonical point order.'''
    # TODO: allow END/x at a point boundary, then y, then an amplitude
    # no larger than the preceding point's amplitude bin. If amplitudes tie,
    # require the new (x,y) pair to follow the previous pair, while ensuring
    # that every unfinished prefix retains at least one legal continuation.
    raise NotImplementedError


@torch.no_grad()
def generate(model, fields, seed, temperature=1.0):
    '''Sample complete grammar-constrained sentences from START.'''
    # TODO: start every row with START; repeatedly mask illegal logits,
    # sample one token, append it, and stop each row at END.
    raise NotImplementedError

In [ ]:
fixed_tensor = field_tensor(fixed_field[None])
sample_fields = fixed_tensor.repeat(200, 1, 1, 1)
sampled_tokens = generate(model, sample_fields, SEED + 5)
model_catalogs = [detokenize_catalog(row, SPEC) for row in sampled_tokens.numpy()]

def sentence_is_valid(sequence, spec):
    sequence = np.asarray(sequence)
    ends = np.flatnonzero(sequence == spec.END)
    if len(ends) != 1 or sequence[0] != spec.START:
        return False
    end = int(ends[0])
    if (end - 1) % 3 or np.any(sequence[end + 1:] != spec.PAD):
        return False
    body = sequence[1:end].reshape(-1, 3)
    if not (
        np.all((0 <= body[:, 0]) & (body[:, 0] < spec.n_bins))
        and np.all((spec.n_bins <= body[:, 1]) & (body[:, 1] < 2 * spec.n_bins))
        and np.all((2 * spec.n_bins <= body[:, 2]) & (body[:, 2] < 3 * spec.n_bins))
    ):
        return False
    previous, following = body[:-1], body[1:]
    amplitude_falls = previous[:, 2] > following[:, 2]
    amplitude_ties = previous[:, 2] == following[:, 2]
    coordinates_follow = (
        (previous[:, 0] < following[:, 0])
        | (
            (previous[:, 0] == following[:, 0])
            & (previous[:, 1] <= following[:, 1])
        )
    )
    return bool(np.all(amplitude_falls | (amplitude_ties & coordinates_follow)))


grammar_valid_fraction = float(np.mean([
    sentence_is_valid(row, SPEC) for row in sampled_tokens.numpy()
]))
assert grammar_valid_fraction == 1.0

truth_rng = np.random.default_rng(SEED + 6)
continuous_truth_catalogs = [
    draw_catalog(fixed_field, truth_rng) for _ in range(200)
]
# Compare the model with the part of the truth its tokens can represent.
truth_catalogs = [
    detokenize_catalog(tokenize_catalog(item, SPEC), SPEC)
    for item in continuous_truth_catalogs
]

In [ ]:
selected = [
    ("field", None),
    ("tokenized truth", truth_catalogs[0]),
    ("model 1", model_catalogs[0]),
    ("model 2", model_catalogs[1]),
    ("model 3", model_catalogs[2]),
    ("model 4", model_catalogs[3]),
]
fig, axes = plt.subplots(2, 3, figsize=(9.5, 6.1), constrained_layout=True)
for ax, (title, catalog) in zip(axes.flat, selected):
    ax.imshow(
        fixed_field, origin="lower",
        extent=(0, SPEC.field_size, 0, SPEC.field_size),
        cmap="magma", alpha=0.8,
    )
    if catalog is not None and len(catalog):
        ax.scatter(catalog[:, 0], catalog[:, 1], s=30 + 80 * catalog[:, 2],
                   c=catalog[:, 2], cmap="viridis", vmin=0.35, vmax=1.0)
    ax.set(title=title if catalog is None else f"{title}: N={len(catalog)}",
           xlim=(0, SPEC.field_size), ylim=(0, SPEC.field_size),
           xlabel="x", ylabel="y")
    ax.set_aspect("equal")
    ax.grid(False)
savefig(fig, "07_sampled_catalogs.png", "trained-teaching-model")
plt.show()

truth_counts = np.array([len(item) for item in truth_catalogs])
model_counts = np.array([len(item) for item in model_catalogs])
truth_points = np.concatenate([item for item in truth_catalogs if len(item)])
model_points = np.concatenate([item for item in model_catalogs if len(item)])

fig, axes = plt.subplots(1, 4, figsize=(13.8, 3.4), constrained_layout=True)
bins = np.arange(-0.5, SPEC.max_points + 1.5)
axes[0].hist(
    truth_counts, bins=bins, density=True, histtype="step", lw=2,
    label="tokenized truth",
)
axes[0].hist(model_counts, bins=bins, density=True, histtype="step", lw=2, label="model")
axes[0].set(
    xticks=range(SPEC.max_points + 1), xlabel="count N",
    ylabel="probability", title="catalog length",
)
axes[0].legend()
amplitude_edges = np.linspace(SPEC.amplitude_min, SPEC.amplitude_max, SPEC.n_bins + 1)
axes[1].hist(
    truth_points[:, 2], bins=amplitude_edges, density=True, histtype="step",
    lw=2, label="tokenized truth",
)
axes[1].hist(
    model_points[:, 2], bins=amplitude_edges, density=True, histtype="step",
    lw=2, label="model",
)
axes[1].set(xlabel="amplitude A", ylabel="density", title="marks")
axes[1].legend()
edges = np.linspace(0, SPEC.field_size, SPEC.n_bins + 1)
truth_map = np.histogram2d(truth_points[:, 1], truth_points[:, 0], bins=edges)[0]
model_map = np.histogram2d(model_points[:, 1], model_points[:, 0], bins=edges)[0]
truth_map /= truth_map.sum()
model_map /= model_map.sum()
vmax = max(truth_map.max(), model_map.max())
for ax, values, title in [
    (axes[2], truth_map, "tokenized truth positions"),
    (axes[3], model_map, "model positions"),
]:
    image = ax.imshow(
        values, origin="lower",
        extent=(0, SPEC.field_size, 0, SPEC.field_size),
                      cmap="magma", vmin=0, vmax=vmax)
    ax.set(xlabel="x", ylabel="y", title=title)
fig.colorbar(image, ax=axes[2:], label="probability per spatial bin")
savefig(fig, "07_conditional_distributions.png", "trained-teaching-model")
plt.show()

def total_variation(first, second):
    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)
    return 0.5 * float(np.abs(first - second).sum())


truth_count_p = np.bincount(truth_counts, minlength=SPEC.max_points + 1) / len(truth_counts)
model_count_p = np.bincount(model_counts, minlength=SPEC.max_points + 1) / len(model_counts)
truth_mark_p = np.histogram(truth_points[:, 2], bins=amplitude_edges)[0]
model_mark_p = np.histogram(model_points[:, 2], bins=amplitude_edges)[0]
truth_mark_p = truth_mark_p / truth_mark_p.sum()
model_mark_p = model_mark_p / model_mark_p.sum()

summary = {
    "evidence_domain": "trained-teaching-model",
    "scope": "pedagogical conditional point process; not GOTHAM and not a halo finder",
    "train_examples": len(train_fields),
    "validation_examples": len(val_fields),
    "epochs_run": len(train_history),
    "best_epoch": best_epoch,
    "validation_loss": best_validation_loss,
    "shuffled_field_loss": best_shuffled_loss,
    "position_only_loss": position_baseline,
    "truth_samples": len(truth_catalogs),
    "model_samples": len(model_catalogs),
    "truth_mean_count": float(truth_counts.mean()),
    "model_mean_count": float(model_counts.mean()),
    "fixed_field_count_tv": total_variation(truth_count_p, model_count_p),
    "fixed_field_position_tv": total_variation(truth_map, model_map),
    "fixed_field_mark_tv": total_variation(truth_mark_p, model_mark_p),
    "comparison_target": "tokenized truth decoded to bin centers",
    "sampling_seed": SEED + 5,
    "sampling_temperature": 1.0,
    "grammar_valid_fraction": grammar_valid_fraction,
    "causal_suffix_invariance": "PASS",
}
(OUTPUT_DIR / "07_toy_catalog_summary.json").write_text(json.dumps(summary, indent=2))
print(
    "fixed-field TV — count: "
    f"{summary['fixed_field_count_tv']:.3f}, position: "
    f"{summary['fixed_field_position_tv']:.3f}, mark: "
    f"{summary['fixed_field_mark_tv']:.3f}"
)

## 6. Validate — routing is not sensitivity

Cross-attention reports where the model routed information.
Occlusion asks how the catalog log probability changes when one
field patch is replaced. The two maps answer different questions,
and the catalog ensemble above remains the external scientific test.

In [ ]:
model.eval()
probe_field = val_fields[:1]
probe_tokens = val_tokens[:1, :-1]
_, weights = model(probe_field, probe_tokens, return_attention=True)
valid_queries = val_tokens[0, 1:].ne(SPEC.PAD)
patch_size = model.patch.kernel_size[0]
patches_per_axis = SPEC.field_size // patch_size
attention_map = (
    weights[0, :, valid_queries]
    .mean(dim=(0, 1))
    .detach()
    .numpy()
    .reshape(patches_per_axis, patches_per_axis)
)

@torch.no_grad()
def sequence_log_probability(field, sequence):
    logits = model(field, sequence[:, :-1])
    targets = sequence[:, 1:]
    chosen = torch.log_softmax(logits, -1).gather(-1, targets[..., None])[..., 0]
    return float(chosen[targets.ne(SPEC.PAD)].sum())

base_logp = sequence_log_probability(probe_field, val_tokens[:1])
occlusion = np.zeros((patches_per_axis, patches_per_axis))
for py in range(patches_per_axis):
    for px in range(patches_per_axis):
        changed = probe_field.clone()
        changed[
            :, :,
            patch_size*py:patch_size*(py+1),
            patch_size*px:patch_size*(px+1),
        ] = probe_field.mean()
        occlusion[py, px] = base_logp - sequence_log_probability(changed, val_tokens[:1])

fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.3), constrained_layout=True)
field_image = axes[0].imshow(
    val_fields_np[0], origin="lower", cmap="magma",
    extent=(0, SPEC.field_size, 0, SPEC.field_size),
)
axes[0].set(title="probe field", xlabel="x", ylabel="y")
axes[0].grid(False)
fig.colorbar(field_image, ax=axes[0])
for ax, values, title, cmap in [
    (axes[1], attention_map, "cross-attention routing", "viridis"),
    (axes[2], occlusion, r"$\log p_{\rm original}-\log p_{\rm occluded}$", "coolwarm"),
]:
    limits = {}
    if cmap == "coolwarm":
        bound = float(np.max(np.abs(values)))
        limits = {"vmin": -bound, "vmax": bound}
    image = ax.imshow(values, origin="lower", cmap=cmap, **limits)
    ax.set(title=title, xlabel="patch x", ylabel="patch y")
    ax.set_xticks(range(patches_per_axis))
    ax.set_yticks(range(patches_per_axis))
    ax.grid(False)
    fig.colorbar(image, ax=ax)
savefig(fig, "07_toy_interpretability.png", "trained-teaching-model")
plt.show()

receipt = {
    "evidence_domain": "trained-teaching-model",
    "cross_attention_map": attention_map.tolist(),
    "occlusion_response": occlusion.tolist(),
    "occlusion_replacement": "within-example global mean fill of one encoder patch",
    "claim_boundary": "attention is routing; occlusion is sensitivity to the stated replacement",
}
(OUTPUT_DIR / "07_interpretability_receipt.json").write_text(json.dumps(receipt, indent=2))

## Takeaway

Teacher-forced loss measures next-token prediction under true
prefixes. Free-running samples test the actual catalog generator.
The shuffled-field control tests whether the reader matters.
Comparing with tokenized truth keeps representation error separate
from model error. This model is still a teaching demonstration, not
a scientifically calibrated catalog generator.

## After the core exercise

Change one choice at a time and keep the evaluation seeds fixed:

1. **Wiring, then data:** first overfit 32 examples with a fresh
   model as a debugging check. Next repeat with 512 training fields
   and checkpoints at 15, 40, and 80 epochs. Then add independent
   fields or redraw toy catalogs from the known process each epoch.
   Decide which change reduces overfitting.
2. **Representation:** change `n_bins` or the encoder `patch_size`.
   Track both the physical resolution and the harder prediction
   problem.
3. **Position and memory:** zero the token or field position
   embeddings, or train with shuffled field/catalog pairs. Compare
   held-out loss and fixed-field spatial maps.
4. **Sampling:** try temperatures 0.7, 1.0, and 1.3 with matched
   seeds. Compare count, position, and mark TV; temperature is not
   a physical uncertainty parameter.
5. **Mechanism:** occlude the brightest patch and a dim patch.
   Check whether the attention ranking predicts the log-probability
   response rather than assuming that it must.